# [Module 05 -- Fundamentals] Research Assistant Crew

> **MLCourse -- Agentic AI -- CrewAI Fundamentals**

> This module puts it all together: a complete multi-agent crew that researches
> a topic, writes a report, and polishes it. Three agents, three tasks,
> sequential process -- the full CrewAI workflow from start to finish.

## What you'll learn

- Designing a multi-agent workflow: researcher, writer, editor.
- Assigning tools to agents (web search + file reading).
- Chaining tasks via `context` so each agent builds on the previous one.
- Running the crew and inspecting the final output.
- Guarding external API calls so the notebook runs without keys.

In [ ]:
# --- Standard library imports -------------------------------------------------
import os
from pathlib import Path

# --- Third-party imports ------------------------------------------------------
from dotenv import load_dotenv

# Walk up to track root.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("Setup complete. Track root:", TRACK)

## 1. Imports

In [ ]:
try:
    from crewai import Agent, Task, Crew, Process
    from langchain_ollama import ChatOllama
    print("[OK] crewai and ChatOllama imported.")
except ImportError as e:
    print("[ERROR] pip install crewai \"crewai[tools]\" langchain-ollama")
    print("        Detail:", e)

In [ ]:
# Import tools -- each guarded so the notebook runs without crewai[tools].
try:
    from crewai_tools import FileReadTool, ScrapeWebsiteTool
    print("[OK] FileReadTool and ScrapeWebsiteTool imported.")
    TOOLS_AVAILABLE = True
except ImportError as e:
    print("[WARN] crewai[tools] not fully installed. Tool demos will be skipped.")
    print("       Run: pip install \"crewai[tools]\"")
    TOOLS_AVAILABLE = False

In [ ]:
LLM_MODEL = "llama3.1:8b"
llm = ChatOllama(model=LLM_MODEL)

try:
    resp = llm.invoke("Reply OK")
    print("[OK] Ollama live:", resp.content[:30])
except Exception as e:
    print("[WARN] Ollama down. Start: ollama pull", LLM_MODEL)
    print("       Detail:", e)

## 2. Prepare sample reference data

We create a small text file that the researcher agent will read as "source
material." This makes the crew self-contained -- no external API keys needed.

In [ ]:
import tempfile

data_dir = Path(tempfile.gettempdir()) / "crewai_research_demo"
data_dir.mkdir(exist_ok=True)

reference_file = data_dir / "reference_notes.txt"
reference_file.write_text(
    "Research Notes: AI Agents in 2026\n"
    "==================================\n"
    "\n"
    "1. Multi-agent frameworks like CrewAI, AutoGen, and LangGraph have\n"
    "   matured significantly. CrewAI focuses on role-based collaboration\n"
    "   with a simple Agent-Task-Crew abstraction.\n"
    "\n"
    "2. Key trend: agents are moving from demos to production. Companies\n"
    "   are deploying customer support agents, code review agents, and\n"
    "   research assistants at scale.\n"
    "\n"
    "3. Tool integration has become standardized. Agents can use web search,\n"
    "   file I/O, database queries, and API calls through unified tool\n"
    "   interfaces.\n"
    "\n"
    "4. The main challenges are: reliability (hallucination prevention),\n"
    "   cost control (token usage), and observability (logging agent\n"
    "   decisions for debugging).\n"
    "\n"
    "5. Open-source models (Llama 3.2, Mistral, Qwen) are now competitive\n"
    "   with proprietary models for many agent tasks, enabling fully\n"
    "   local and private deployments.\n"
)

print("Reference file created:", reference_file)
print("Contents:\n", reference_file.read_text()[:200], "...")

## 3. Define the three agents

A research assistant crew needs three specialized roles:

1. **Researcher** -- reads source material and web content, gathers facts.
2. **Writer** -- synthesizes research into a coherent report.
3. **Editor** -- polishes the report for clarity, grammar, and flow.

In [ ]:
# Agent 1: Researcher -- reads files and scrapes the web for information.
researcher = Agent(
    role="Senior Research Analyst",
    goal=(
        "Gather comprehensive, accurate information on the given topic "
        "from reference materials and web sources."
    ),
    backstory=(
        "You are a seasoned research analyst with expertise in AI and "
        "technology. You are meticulous about sourcing and always cite "
        "your references. You read files carefully and extract key facts."
    ),
    llm=llm,
    tools=[],
    allow_delegation=False,
    max_iter=8,
    verbose=True,
)

# Add tools only if crewai[tools] is available.
if TOOLS_AVAILABLE:
    researcher.tools = [FileReadTool(), ScrapeWebsiteTool()]
    print("Researcher assigned: FileReadTool, ScrapeWebsiteTool")
else:
    print("Researcher: no tools (will work from description only)")

In [ ]:
# Agent 2: Writer -- synthesizes research into a structured report.
writer = Agent(
    role="Technical Writer",
    goal=(
        "Transform raw research findings into a well-structured, "
        "engaging report that is clear and accessible."
    ),
    backstory=(
        "You are a skilled technical writer who excels at turning complex "
        "topics into readable content. You use clear headings, short "
        "paragraphs, and concrete examples."
    ),
    llm=llm,
    tools=[],
    allow_delegation=False,
    max_iter=5,
    verbose=True,
)

In [ ]:
# Agent 3: Editor -- polishes the final report.
editor = Agent(
    role="Senior Editor",
    goal=(
        "Polish the report for grammar, clarity, flow, and conciseness. "
        "Ensure the final output is publication-ready."
    ),
    backstory=(
        "You are a meticulous editor with decades of experience in "
        "technical publishing. You catch awkward phrasing, fix grammar, "
        "and tighten prose without losing meaning."
    ),
    llm=llm,
    tools=[],
    allow_delegation=False,
    max_iter=3,
    verbose=True,
)

print("All 3 agents defined:")
print("  1.", researcher.role)
print("  2.", writer.role)
print("  3.", editor.role)

## 4. Define the three tasks

Each task builds on the previous one through the `context` parameter. The
researcher's output feeds into the writer's task, and the writer's output
feeds into the editor's task.

In [ ]:
# Task 1: Research -- read the reference file and summarize findings.
research_task = Task(
    description=(
        f"Read the reference file at {reference_file} and summarize the "
        "key findings about AI agents in 2026. Organize the summary into "
        "3-5 bullet points covering: frameworks, production adoption, tool "
        "integration, challenges, and open-source models."
    ),
    expected_output=(
        "A structured summary with 3-5 bullet points, each containing "
        "a specific fact or insight from the reference material."
    ),
    agent=researcher,
)

In [ ]:
# Task 2: Write -- turn the research summary into a full report.
write_task = Task(
    description=(
        "Using the research findings provided, write a 300-400 word report "
        "titled 'AI Agents in 2026: A State of the Field Overview'. "
        "Structure it with an introduction, 3 main sections (frameworks, "
        "production adoption, challenges), and a brief conclusion."
    ),
    expected_output=(
        "A complete report in markdown format with a title, introduction, "
        "3 sections with headings, and a conclusion. 300-400 words."
    ),
    agent=writer,
    context=[research_task],        # Writer sees the researcher's output.
)

In [ ]:
# Task 3: Edit -- polish the draft report.
edit_task = Task(
    description=(
        "Review and polish the report for: grammar and spelling, "
        "sentence flow and readability, conciseness (remove filler words), "
        "and overall structure. Return the final edited version."
    ),
    expected_output=(
        "The final polished report in markdown format, ready for "
        "publication. Same structure as the input but cleaner."
    ),
    agent=editor,
    context=[write_task],           # Editor sees the writer's output.
)

print("All 3 tasks defined with context chaining:")
print("  research_task -> write_task (context) -> edit_task (context)")

## 5. Build and run the crew

We assemble the crew with `Process.sequential` so tasks run in order:
research, then write, then edit. The context chain ensures each agent
sees the previous agent's output.

In [ ]:
research_crew = Crew(
    agents=[researcher, writer, editor],
    tasks=[research_task, write_task, edit_task],
    process=Process.sequential,
    verbose=True,                   # See the full execution flow.
)

print("Crew assembled:")
print("  Agents:", [a.role for a in research_crew.agents])
print("  Tasks:", len(research_crew.tasks))
print("  Process:", research_crew.process)

## 6. Kickoff -- run the full pipeline

`crew.kickoff()` executes all three tasks sequentially. Each agent sees the
prior agent's output through the context chain. The first call may be slow
due to model warm-up; subsequent calls are faster.

In [ ]:
try:
    final_result = research_crew.kickoff()
    print("\n" + "=" * 60)
    print("=== FINAL REPORT ===")
    print("=" * 60)
    print(final_result.raw)
    print("=" * 60)
except Exception as e:
    print("[demo skipped] Ensure Ollama is running with model:", LLM_MODEL)
    print("       Detail:", e)

## 7. Inspect the output

The `CrewOutput` object provides structured access to the result, token
usage, and any metadata.

In [ ]:
try:
    print("Output type    :", type(final_result).__name__)
    print("Raw length     :", len(final_result.raw), "chars")
    print()
    print("Token usage:")
    for key, val in final_result.token_usage.items():
        print(f"  {key}: {val}")
except NameError:
    print("[skipped] Run section 6 first.")

## 8. Save the report to disk

Write the final report to a file for later use. This also demonstrates
how to extract and persist crew output in a real workflow.

In [ ]:
report_output = data_dir / "final_report.md"

try:
    report_output.write_text(final_result.raw, encoding="utf-8")
    print("Report saved to:", report_output)
    print("File size:", report_output.stat().st_size, "bytes")
    print()
    print("=== Saved Report (preview) ===")
    print(report_output.read_text(encoding="utf-8")[:500])
except NameError:
    print("[skipped] Run section 6 first to create 'final_result'.")

## 9. Variant: crew without external tools

If you do not have `crewai[tools]` installed or want to avoid external API
calls, you can run the same crew with no tools. The agents rely solely on
the LLM's internal knowledge and the reference data passed in the task
description.

In [ ]:
# Self-contained variant -- no tools, no API keys needed.
researcher_basic = Agent(
    role="Researcher",
    goal="Summarize key facts about AI agents.",
    backstory="You are a knowledgeable AI researcher.",
    llm=llm,
    allow_delegation=False,
    verbose=False,
)

writer_basic = Agent(
    role="Writer",
    goal="Write a clear report from research notes.",
    backstory="You write accessible technical content.",
    llm=llm,
    allow_delegation=False,
    verbose=False,
)

editor_basic = Agent(
    role="Editor",
    goal="Polish the report for clarity.",
    backstory="You are a careful editor.",
    llm=llm,
    allow_delegation=False,
    verbose=False,
)

research_basic = Task(
    description=(
        "Based on your knowledge, summarize the current state of AI agent "
        "frameworks in 2026. Cover: CrewAI, AutoGen, LangGraph. List 3 "
        "key developments for each."
    ),
    expected_output="A structured summary with 3 points per framework.",
    agent=researcher_basic,
)

write_basic = Task(
    description=(
        "Write a 200-word comparison of the three major AI agent "
        "frameworks based on the research provided."
    ),
    expected_output="A concise comparison in markdown.",
    agent=writer_basic,
    context=[research_basic],
)

edit_basic = Task(
    description="Polish the comparison for grammar and flow.",
    expected_output="The final edited comparison.",
    agent=editor_basic,
    context=[write_basic],
)

basic_crew = Crew(
    agents=[researcher_basic, writer_basic, editor_basic],
    tasks=[research_basic, write_basic, edit_basic],
    process=Process.sequential,
    verbose=False,                  # Quiet mode for this variant.
)

try:
    basic_result = basic_crew.kickoff()
    print("=== Basic Crew Result (no tools) ===")
    print(basic_result.raw[:500])
except Exception as e:
    print("[demo skipped]", e)

## 10. Key takeaways

| Component   | Role in this crew                                  |
|-------------|----------------------------------------------------|
| Researcher  | Reads reference files + scrapes web, gathers facts |
| Writer      | Synthesizes research into a structured report      |
| Editor      | Polishes for grammar, flow, and conciseness        |
| Process     | Sequential -- each task sees prior task output      |
| Context     | `[prior_task]` chains task outputs together        |

- A research crew needs at minimum: gather, synthesize, polish.
- `context=[prior_task]` is the key mechanism for information flow.
- Tools (FileReadTool, ScrapeWebsiteTool) are optional but powerful.
- The crew runs with a single `crew.kickoff()` call.
- All external API calls are guarded with try/except.
- The same pattern scales: add more agents, more tasks, more tools.